## Download real datasets

In [1]:
import os, openml, tqdm, json

In [2]:
import numpy as np
import pandas as pd

In [3]:
from pathlib import Path
from openml.tasks import TaskType
from sklearn.preprocessing import LabelEncoder

## Loading TabArena

In [4]:
benchmark = openml.study.get_suite('tabarena-v0.1')
tasks = openml.tasks.list_tasks(task_id=benchmark.tasks, output_format='dataframe')
tasks = tasks[tasks['ttid'] == TaskType.SUPERVISED_CLASSIFICATION]

In [5]:
tasks_tabarena = tasks['tid'].astype(int).tolist()
names_tabarena = tasks['name'].astype(str).tolist()
tasks_tabarena[:5], names_tabarena[:5]

([363613, 363614, 363616, 363618, 363619],
 ['Amazon_employee_access',
  'anneal',
  'APSFailure',
  'bank-marketing',
  'Bank_Customer_Churn'])

In [6]:
def load_tabarena(tid):
    task = openml.tasks.get_task(tid)
    X, y, _, _ = task.get_dataset().get_data(target=task.target_name)
    return X, LabelEncoder().fit_transform(y)

In [7]:
for tid, name in tqdm.tqdm(zip(tasks_tabarena, names_tabarena), total=len(tasks_tabarena)):
    X, y = load_tabarena(tid)
    X.assign(target=y).to_csv(f'../data/processed/tabarena/{name}.csv', index=False)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 38/38 [00:15<00:00,  2.47it/s]


## Loading Talent

In [8]:
names_talent = os.listdir('../data/raw/')
names_talent[:5]

['Performance-Prediction',
 'Contaminant-detection-in-packaged-cocoa-hazelnut-spread-jars-using-Microwaves-Sensing-and-Machine-Learning-9.5GHz(Urbinati)',
 'philippine',
 'Fitness_Club_c',
 'dry_bean_dataset']

In [9]:
def load_split(tid, split):
    blocks = []
    for prefix in ['N', 'C']:
        path = f'../data/raw/{tid}/{prefix}_{split}.npy'
        if os.path.exists(path):
            blocks.append(np.load(path, allow_pickle=True))
    X = np.concatenate(blocks, axis=1)
    y = np.load(f'../data/raw/{tid}/y_{split}.npy', allow_pickle=True)
    return X, y

In [10]:
def load_talent(tid):
    
    Xs, ys = [], []
    for split in ['train', 'test', 'val']:
        try:
            X, y = load_split(tid, split)
            Xs.append(X)
            ys.append(y)
        except:
            continue
     
    X = np.concatenate(Xs, axis=0)
    y = np.concatenate(ys, axis=0).ravel()
    return X, LabelEncoder().fit_transform(y)

In [11]:
for name in tqdm.tqdm(names_talent):
    info = json.loads(Path(f'../data/raw/{name}/info.json').read_text())
    if info['task_type'] in ['binclass', 'multiclass']:
        X, y = load_talent(name)
        pd.DataFrame(X).assign(target=y).to_csv(f'../data/processed/talent/{name}.csv', index=False)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 300/300 [00:17<00:00, 17.42it/s]


In [12]:
len(os.listdir('../data/processed/tabarena')), len(os.listdir('../data/processed/talent'))

(38, 200)

## Removing duplicates

In [13]:
lower_tabarena = {n.lower() for n in names_tabarena}

In [14]:
for name in names_talent:
    if name.lower() in lower_tabarena:
        path = f'../data/processed/talent/{name}.csv'
        if os.path.exists(path):
            os.remove(path)

## Final result

In [15]:
len(os.listdir('../data/processed/tabarena')), len(os.listdir('../data/processed/talent'))

(38, 183)